# Stable Audio 3 Small Music — Game Music Quality Pilot

**Purpose:** test whether a dedicated, commercially usable music checkpoint can produce a polished, usable background-music bed for a game scene.

This is a small inference experiment only: **no training, no fine-tuning, no video input, and no external generation API**. We keep one prompt fixed and generate three seeds so we can judge musical quality and variation without changing several things at once.

The outputs are 30-second instrumental candidates. This notebook is a music-layer experiment alongside the existing Kandinsky video-to-audio work; it does not replace that workflow or test synchronization.

## Before running: model access and commercial terms

1. Accept the access and license conditions on the [Stable Audio 3 Small Music model page](https://huggingface.co/stabilityai/stable-audio-3-small-music). The page also requires acceptance of the [Gemma Terms of Use](https://ai.google.dev/gemma/terms) for its text encoder.
2. Add your Hugging Face **read token** to Colab Secrets as `HF_TOKEN`, and enable notebook access. Do not paste the token into a cell or print it.
3. Stability AI's Community License permits qualifying commercial use below its $1M annual-revenue threshold; Stability says commercial product/service research must be registered. Review the current [license and registration requirements](https://stability.ai/license) before product integration.

The notebook records the model revision and generation settings. It does not establish legal clearance.

## Step 1 — choose a Colab GPU runtime

Use **Runtime → Change runtime type → A100** if available. The Small Music model is much lighter than the video model, but the first run still downloads the environment and gated weights. Subsequent generation cells reuse the loaded model.

In [ ]:
import os, shutil, subprocess, sys, time, json, textwrap
from pathlib import Path

print("Python:", sys.version.split()[0])
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    )
    print("GPU:", result.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU not detected. Select a GPU runtime in Colab before continuing.")

## Step 2 — install the official inference package in an isolated Python environment

The setup uses the official Stable Audio 3 repository and its lockfile inside a separate Python 3.10 environment. That keeps Colab's own notebook packages intact and avoids mixing PyTorch versions with the Kandinsky environment.

In [ ]:
REPO = Path("/content/stable-audio-3")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-audio-3.git", str(REPO)],
        check=True
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was installed but is not available on PATH. Restart the runtime and rerun this setup cell.")

subprocess.run([UV, "python", "install", "3.10"], check=True)
subprocess.run([UV, "sync", "--project", str(REPO), "--python", "3.10"], check=True)

PYTHON = REPO / ".venv" / "bin" / "python"
if not PYTHON.exists():
    raise FileNotFoundError(f"Isolated Python environment was not created: {PYTHON}")

REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Stable Audio 3 commit:", REVISION)
print("Environment ready:", PYTHON)

## Step 3 — configure one scene and keep it fixed

This baseline prompt describes an **instrumental background score**, not a sound effect. You can edit the prompt to match a scene you want to test. Keep it unchanged while comparing the three seeds. The prompt is in English because the model card lists English conditioning.

In [ ]:
PROMPT = (
    "Instrumental cinematic ambient game music for exploring an ancient forest at dawn; "
    "gentle flowing string motif, light harp, warm sustained pads, calm and curious mood, "
    "natural dynamics, restrained background score. No vocals, no lyrics, no percussion, no sound effects."
)
DURATION_SECONDS = 30
SEEDS = [42, 123, 7]
STEPS = 8

print("Candidates:", len(SEEDS), "| duration:", DURATION_SECONDS, "seconds")
print("Prompt:", PROMPT)

## Step 4 — load the Small Music checkpoint

The model is gated. The cell reads `HF_TOKEN` from Colab Secrets without displaying it. If access fails, confirm that you accepted the model page's conditions and enabled notebook access for the secret.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN
del HF_TOKEN

check = subprocess.run(
    [str(PYTHON), "-c", "import torch; print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'"],
    text=True, capture_output=True
)
print(check.stdout)
if check.returncode:
    print(check.stderr[-3000:])
    raise RuntimeError("GPU/PyTorch preflight failed.")

print("HF token is available to the model loader (value hidden).")

## Step 5 — generate three candidates

Only the seed changes between candidates. They run one at a time to keep peak memory low. Each WAV is saved at the model's native sample rate without extra normalization or post-processing.

In [ ]:
OUT_DIR = Path("/content/stable_audio_music_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)

generation_script = f"""
import json, time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from stable_audio_3 import StableAudioModel

prompt = {json.dumps(PROMPT)}
duration = {int(DURATION_SECONDS)}
seeds = {json.dumps(SEEDS)}
steps = {int(STEPS)}
out_dir = Path({json.dumps(str(OUT_DIR))})
out_dir.mkdir(parents=True, exist_ok=True)

model = StableAudioModel.from_pretrained("small-music", device="cuda", model_half=True)
sample_rate = int(model.model.sample_rate)
manifest = {{
    "schema": "stable-audio-3-small-music-pilot-v1",
    "model": "stabilityai/stable-audio-3-small-music",
    "repo_revision": {json.dumps(REVISION)},
    "prompt": prompt,
    "duration_seconds": duration,
    "steps": steps,
    "cfg_scale": 1.0,
    "seeds": seeds,
    "sample_rate_hz": sample_rate,
    "outputs": []
}}

for seed in seeds:
    started = time.time()
    audio = model.generate(
        prompt=prompt,
        duration=duration,
        steps=steps,
        cfg_scale=1.0,
        seed=int(seed),
        batch_size=1,
    )
    waveform = audio[0].detach().float().cpu().numpy().T
    filename = f"candidate_seed_{{seed}}.wav"
    output_path = out_dir / filename
    sf.write(output_path, waveform, sample_rate, subtype="PCM_16")
    elapsed = time.time() - started
    manifest["outputs"].append({{
        "file": filename,
        "seed": int(seed),
        "duration_seconds": round(len(waveform) / sample_rate, 3),
        "generation_seconds": round(elapsed, 2)
    }})
    print(f"Saved {{filename}} | {{len(waveform) / sample_rate:.2f}} s | {{elapsed:.1f}} s")
    del audio, waveform
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(out_dir / "pilot_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Sample rate:", sample_rate, "Hz")
"""

SCRIPT_PATH = Path("/content/run_stable_audio_music_pilot.py")
SCRIPT_PATH.write_text(generation_script, encoding="utf-8")
subprocess.run([str(PYTHON), str(SCRIPT_PATH)], check=True)

## Step 6 — listen and rate the candidates

Listen at a comfortable volume. Score each item from 1 (poor) to 5 (strong):

- **Musical quality:** does it sound intentional and coherent?
- **Scene fit:** does it suit the described game scene?
- **Background usability:** does it stay behind gameplay rather than demand attention?
- **Artifacts:** audible glitches, harshness, abrupt changes, or unwanted voice/percussion?
- **Variation:** do the seeds produce meaningfully different arrangements?

This pilot answers whether this model can produce a promising standalone music bed. It does **not** yet test video understanding, automatic prompt writing, looping, or synchronization.

In [ ]:
from IPython.display import Audio, display

for seed in SEEDS:
    wav_path = OUT_DIR / f"candidate_seed_{seed}.wav"
    print(f"Candidate — seed {seed}")
    display(Audio(filename=str(wav_path)))

## Step 7 — download the review package

The ZIP contains only the generated WAV files and the manifest. The manifest records the prompt, seeds, model revision, sample rate, duration, and generation time. It contains no access token.

In [ ]:
import zipfile

ZIP_PATH = Path("/content/stable_audio_music_pilot.zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Review package:", ZIP_PATH, "| size:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2), "MB")
from google.colab import files
files.download(str(ZIP_PATH))